In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

In [2]:
df_global = pd.read_csv('data/BostonHousing.csv')

df_global = df_global.rename(columns={
    'crim': 'crime_rate',           # Уровень преступности
    'zn': 'large_lots',              # Доля земли под большие участки
    'indus': 'industry',             # Доля промышленности
    'chas': 'river',                  # У реки (Charles river)
    'nox': 'nox_conc',       # Концентрация оксидов азота
    'rm': 'rooms_avg',                # Среднее число комнат
    'age': 'old_buildings',           # Доля старых домов (до 1940)
    'dis': 'center_dist',  # Расстояние до центров занятости
    'rad': 'highway',          # Доступ к шоссе
    'tax': 'property',            # Налог на недвижимость
    'ptratio': 'pupil_teacher', # Соотношение ученики/учитель
    'b': 'black_population',    # Индекс чернокожего населения 
    'lstat': 'lower_status',  # % населения с низким статусом
    'medv': 'price'                    # ЦЕЛЬ: медианная цена дома
})


In [3]:
# Целевая переменная
X = df_global.drop('price', axis=1) # все столбцы, кроме price. axis=1 - удаляем столбец, 0 - удаляем строку
y = df_global['price'] # только price

# 1. Разделяем данные
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
) # 0.3 данных - тест, random_state - число для рандома

# 2. Считаем медианы по каждому столбцу НА ТРЕНИРОВОЧНЫХ данных
medians = X_train.median()

# 3. Заполняем пропуски в train
X_train_filled = X_train.fillna(medians) # fillna - заплатка

# 4. Заполняем пропуски в test (ТЕМИ ЖЕ медианами!)
X_test_filled = X_test.fillna(medians) # если заполнить медианами из теста, то будет утечка данных, потому что мы будем использоваться информацию, которая нам неизвестна на момент обучения модели

# 5. Проверяем, что пропусков больше нет
# print(f"\nПропуски в train после заполнения: {X_train_filled.isnull().sum().sum()}")
# print(f"Пропуски в test после заполнения: {X_test_filled.isnull().sum().sum()}")

# 6. Обучаем модель
model = LinearRegression()
model.fit(X_train_filled, y_train) # X_train_filled - учебник, y_train - ответы

# 7. Оцениваем качество
train_score = model.score(X_train_filled, y_train)
raw_score = model.score(X_test_filled, y_test)

print(f"\nR² на train: {train_score:.3f}")
print(f"R² на test:  {raw_score:.3f}")


R² на train: 0.743
R² на test:  0.710


In [4]:
df_improved = df_global.copy()

In [5]:
df_improved['crime_rate_log'] = np.log1p(df_improved['crime_rate']) # ужасный хвост, точно логарифмируем
df_improved['lower_status_log'] = np.log1p(df_improved['lower_status']) # сильная корреляция с ценой, поэтому нужно логарифмировать
df_improved.drop('lower_status', axis=1, inplace=True) # удаляем, потому что мы уже закодировали его в логарифм

In [6]:
# df_improved.drop('river', axis=1, inplace=True)
avg_river_price = df_global[df_global['river'] == 1]['price'].mean()
print(f"Средняя цена домов у реки: {avg_river_price:.2f}")
avg_without_river_price = df_global[df_global['river'] == 0]['price'].mean()
print(f"Средняя цена домов без реки: {avg_without_river_price:.2f}")

Средняя цена домов у реки: 28.44
Средняя цена домов без реки: 22.09


In [7]:
df_improved['many_rooms'] = (df_improved['rooms_avg'] > 7).astype(int)
df_improved.drop('rooms_avg', axis=1, inplace=True) 

In [8]:
df_improved['center_dist_log'] = np.log1p(df_improved['center_dist'])

In [9]:
df_improved.drop('black_population', axis=1, inplace=True) 

In [10]:
X_imp = df_improved.drop('price', axis=1)
y_imp = np.log1p(df_improved['price'])

X_train_imp, X_test_imp, y_train_imp, y_test_imp = train_test_split(
    X_imp, y_imp, test_size=0.3, random_state=42
)

In [11]:
crime_rate_95_percentile = X_train_imp['crime_rate'].quantile(0.95)
X_train_imp['high_crime'] = (X_train_imp['crime_rate'] > crime_rate_95_percentile).astype(int)
X_test_imp['high_crime'] = (X_test_imp['crime_rate'] > crime_rate_95_percentile).astype(int)

In [12]:
old_buildings_80_percentile = X_train_imp['old_buildings'].quantile(0.8)
X_train_imp['is_old_building'] = (X_train_imp['old_buildings'] > old_buildings_80_percentile).astype(int)
X_test_imp['is_old_building'] = (X_test_imp['old_buildings'] > old_buildings_80_percentile).astype(int)

In [13]:
nox_conc_80_percentile = X_train_imp['nox_conc'].quantile(0.8)
X_train_imp['nox_conc_high'] = (X_train_imp['nox_conc'] > nox_conc_80_percentile).astype(int)
X_test_imp['nox_conc_high'] = (X_test_imp['nox_conc'] > nox_conc_80_percentile).astype(int)
X_train_imp.drop('nox_conc', axis=1, inplace=True)
X_test_imp.drop('nox_conc', axis=1, inplace=True)

In [14]:
lower_status_log_89_percentile = X_train_imp['lower_status_log'].quantile(0.89)
X_train_imp['lower_status_log_high'] = (X_train_imp['lower_status_log'] > lower_status_log_89_percentile).astype(int)
X_test_imp['lower_status_log_high'] = (X_test_imp['lower_status_log'] > lower_status_log_89_percentile).astype(int)

In [15]:
pupil_teacher_89_percentile = X_train_imp['pupil_teacher'].quantile(0.7)
X_train_imp['pupil_teacher_high'] = (X_train_imp['pupil_teacher'] > pupil_teacher_89_percentile).astype(int)
X_test_imp['pupil_teacher_high'] = (X_test_imp['pupil_teacher'] > pupil_teacher_89_percentile).astype(int)

In [16]:
medians = X_train_imp.median()
X_train_imp = X_train_imp.fillna(medians)
X_test_imp = X_test_imp.fillna(medians)

# это нужно для того, чтобы признаки были в одном масштабе, иначе модель может плохо обучиться, потому что будет думать, что признаки с большим масштабом важнее, чем признаки с маленьким масштабом
scaler = StandardScaler() 
X_train_imp = scaler.fit_transform(X_train_imp)
X_test_imp = scaler.transform(X_test_imp)

model_imp = LinearRegression()
model_imp.fit(X_train_imp, y_train_imp)

imp_score = model_imp.score(X_test_imp, y_test_imp)

In [17]:
print(f"{'Модель':<30} {'R²':<10}")
print(f"{'Базовая':<30} {raw_score:.3f}")
print(f"{'Улучшенная':<30} {imp_score:.3f}")
print(f"{'Прирост':<30} {imp_score - raw_score:+.5f}")

Модель                         R²        
Базовая                        0.710
Улучшенная                     0.848
Прирост                        +0.13828
